# Step 7c: retrain the classifiers and the detector at 10 m

Steps B and C of the Sentinel-2 experiment. ShipRSImageNet is resampled to 10 m per pixel, Sentinel-2's resolution, and everything is trained again on the result:

1. **Classifiers.** The small CNN, the VGG-style CNN and the ResNet101 at label levels 1, 2 and 3, on vessel crops cut at a fixed scale, so that a longer vessel is a longer shape in the crop. Vessels shorter than 20 m are left out. Each run is selected on validation and scored once on the held-out test split.
2. **Detector.** YOLO11s-OBB on 320 pixel canvases composed from many resampled scenes, scored once on test canvases.

Each image is resampled by its own factor. Its resolution comes from step 7b: measured from vessels of known length where possible, otherwise its source's median, otherwise the recorded value.

Runtime > Change runtime type > GPU before running. If the session drops, run all cells again: finished runs are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings

In [ ]:
REPO_URL = 'https://github.com/bprtdaniel/vessel-watch.git'
BRANCH = 'main'

DATA_DIR_DRIVE = '/content/drive/MyDrive/5.Projects/Data/ShipRSImageNet_V1'
DATA_DIR_LOCAL = '/content/data/ShipRSImageNet_V1'
RUNS_DIR = '/content/drive/MyDrive/5.Projects/runs'  # results are written straight to Drive
CACHE_DIR = '/content/cache'                         # resampled crops and canvases, rebuilt each session
NUM_WORKERS = 8

## Code

In [ ]:
import os, subprocess

REPO_DIR = '/content/vessel-watch'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

# Colab already ships torch, torchvision, pandas, scikit-learn, matplotlib, pyyaml and opencv
%pip install -q --no-deps -e /content/vessel-watch
%pip install -q ultralytics
%cd /content/vessel-watch
!git log --oneline -1

## Data

Reading images from Drive during training is slow, so they are copied to the Colab disk once per session.

In [ ]:
import shutil

if os.path.exists(DATA_DIR_LOCAL):
    print('Local copy already exists')
else:
    os.makedirs(os.path.dirname(DATA_DIR_LOCAL), exist_ok=True)
    shutil.copytree(DATA_DIR_DRIVE, DATA_DIR_LOCAL)
    print('Copied dataset to', DATA_DIR_LOCAL)

os.environ['VESSELWATCH_DATA'] = DATA_DIR_LOCAL
os.environ['VESSELWATCH_RUNS'] = RUNS_DIR
os.environ['VESSELWATCH_CACHE'] = CACHE_DIR

## 1. Classifiers at 10 m

Nine runs, written to `RUNS_DIR/crops10m_level{1,2,3}_{net,net_max,resnet101}/`. The table at the end is also saved as `study_summary_10m.json`.

In [ ]:
!python -m vesselwatch.study --configs configs/crops_10m --num-workers $NUM_WORKERS --summary-name study_summary_10m.json

## 2. Detector at 10 m

The first lines report, per split, how many scenes and vessels went into how many canvases. Results go to `RUNS_DIR/yolo11s_obb_10m/`; the last line gives precision, recall and mAP on the test canvases.

In [ ]:
!python -m vesselwatch.detect.yolo --config configs/detect/yolo11s_obb_10m.yaml

## Results, short enough to copy

Test accuracy by vessel length for the ResNet101 runs, and the detector's test metrics.

In [ ]:
import json
from pathlib import Path
RUNS = Path(RUNS_DIR)

for level in (1, 2, 3):
    f = RUNS / f'crops10m_level{level}_resnet101' / 'metrics_test.json'
    if f.exists():
        m = json.loads(f.read_text())
        by_length = {k: (v['n'], round(v['accuracy'], 1)) for k, v in m['by_length'].items()}
        print(f"level {level} ResNet101: n={m['n']} acc={m['accuracy']:.1f} macro-F1={m['macro_f1']:.1f} "
              f"baseline={m['majority_baseline']['accuracy']:.1f} | by length (n, acc): {by_length}")

det = RUNS / 'yolo11s_obb_10m' / 'metrics_test.json'
print('detector at 10 m (test):', json.loads(det.read_text()) if det.exists() else 'not finished')